# Order Cancellation Prediction

**Projeto de portfólio — Machine Learning / Ciência de Dados**

Previsão de cancelamento de pedidos de delivery a partir de dados sintéticos.

**Autor:** Rodrigo Catão  
**Objetivo:** construir, avaliar e documentar modelos de classificação binária sob desbalanceamento de classes, com foco em prevenção de data leakage e métricas adequadas ao problema.

---

## 1. Introdução

O cancelamento de pedidos gera impacto operacional e financeiro em plataformas de delivery. Este notebook investiga se, com informações disponíveis no momento do pedido, é possível identificar pedidos com maior probabilidade de cancelamento.

**Problema:** classificação binária  
- `0` = pedido entregue  
- `1` = pedido cancelado  

**Cuidados centrais:**
- Desbalanceamento forte (apenas ~8,6% de cancelamentos)
- Evitar data leakage (não usar informações pós-evento)
- Avaliar com Precision, Recall, F1, PR-AUC e matriz de confusão — não apenas acurácia


## 2. Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, classification_report
)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

print('Bibliotecas carregadas.')


## 3. Carregamento dos dados

O dataset deve estar em `data/orders.csv`.  
Obtenha o arquivo em: [mindweave/food-delivery-orders](https://huggingface.co/datasets/mindweave/food-delivery-orders)  
Arquivo: `data/orders.csv` no repositório do Hugging Face.


In [ ]:
# Ajuste o caminho se necessário
DATA_PATH = '../data/orders.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head()


## 4. Análise exploratória

In [ ]:
print('Tipos de dados:')
print(df.dtypes)
print('\nValores ausentes:')
print(df.isnull().sum())
print('\nDuplicados:', df.duplicated().sum())


In [ ]:
# Distribuição da variável-alvo
print(df['status'].value_counts())
print()
print(df['status'].value_counts(normalize=True).round(4))

fig, ax = plt.subplots(figsize=(6, 4))
counts = df['status'].value_counts()
bars = ax.bar(['delivered', 'cancelled'], counts.values, color=['#2ecc71', '#e74c3c'])
ax.set_title('Distribuição das Classes')
ax.set_ylabel('Quantidade')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 20, f'{v}\n({v/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Motivos de cancelamento (apenas para EDA — não usar como feature)
print('Motivos de cancelamento:')
print(df.loc[df['status']=='cancelled', 'cancellation_reason'].value_counts())


In [ ]:
# Análise temporal exploratória
df['order_timestamp'] = pd.to_datetime(df['order_timestamp'])
df['order_hour'] = df['order_timestamp'].dt.hour
df['order_weekday'] = df['order_timestamp'].dt.weekday

hourly = df.groupby('order_hour').agg(
    total=('status', 'count'),
    cancelled=('status', lambda x: (x=='cancelled').sum())
)
hourly['cancel_rate'] = hourly['cancelled'] / hourly['total']
print('Taxa de cancelamento por hora (top 5):')
print(hourly.sort_values('cancel_rate', ascending=False).head())


## 5. Limpeza e preparação

- Criação do alvo `cancelled`
- Seleção de features disponíveis no momento do pedido
- Exclusão de variáveis com risco de leakage


In [ ]:
df['cancelled'] = (df['status'] == 'cancelled').astype(int)

# Features candidatas (sem leakage)
FEATURES = [
    'item_count',
    'subtotal',
    'delivery_fee',
    'estimated_delivery_min',
    'order_hour',
    'order_weekday'
]

# Variáveis excluídas e motivo:
# - actual_delivery_min: só existe após a entrega
# - cancellation_reason: só existe após o cancelamento
# - rating: avaliação posterior
# - id, customer_id, restaurant_id, driver_id: identificadores / disponibilidade questionável no momento do pedido
# - tip, total: podem não estar finalizados ou ser redundantes no instante da previsão

X = df[FEATURES].copy()
y = df['cancelled'].copy()

print('Features utilizadas:', FEATURES)
print('Shape X:', X.shape)
print('Distribuição y:')
print(y.value_counts())


## 6. Engenharia de atributos

Neste experimento inicial não foram criadas features derivadas adicionais além de `order_hour` e `order_weekday` extraídas do timestamp.  
Possíveis extensões futuras: faixas de valor, interações, encoding de restaurante etc. (com cuidado para leakage e cardinalidade).


## 7. Divisão dos dados

Estratégia:
1. Split estratificado 80/20 → treino+val / teste
2. Do treino+val, split estratificado 75/25 → treino / validação

O conjunto de teste fica reservado. Thresholds são escolhidos **somente na validação**.


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f'Treino:     {len(X_train)} (cancelados: {y_train.sum()})')
print(f'Validação:  {len(X_val)} (cancelados: {y_val.sum()})')
print(f'Teste:      {len(X_test)} (cancelados: {y_test.sum()})')


## 8. Baseline — Dummy Classifier

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)

y_dummy = dummy.predict(X_test)
print('Baseline (most_frequent) no teste:')
print(f'  Accuracy: {accuracy_score(y_test, y_dummy):.4f}')
print('  (sempre prevê a classe majoritária — não identifica cancelamentos)')


## 9. Regressão Logística

Pipeline com `StandardScaler` + `LogisticRegression(class_weight='balanced')` para evitar vazamento de escala e lidar com desbalanceamento.


In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000
    ))
])

lr_pipe.fit(X_train, y_train)
print('Regressão Logística treinada.')


## 10. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print('Random Forest treinado.')


## 11. Avaliação dos modelos (threshold = 0,50)

In [ ]:
def evaluate_model(model, X, y, name, threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= threshold).astype(int)
    metrics = {
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'pr_auc': average_precision_score(y, proba),
        'roc_auc': roc_auc_score(y, proba),
    }
    cm = confusion_matrix(y, pred)
    print(f'=== {name} (threshold={threshold:.4f}) ===')
    for k, v in metrics.items():
        print(f'  {k:12s}: {v:.4f}')
    print('  Matriz de confusão:')
    print(cm)
    return metrics, proba, cm

print('--- Validação ---')
lr_m05, lr_proba_val, lr_cm05 = evaluate_model(lr_pipe, X_val, y_val, 'Logistic Regression')
print()
rf_m05, rf_proba_val, rf_cm05 = evaluate_model(rf, X_val, y_val, 'Random Forest')


## 12. Ajuste de threshold na validação

Busca do threshold que maximiza o F1-score **somente na validação**.  
O conjunto de teste não é usado para escolher o threshold.


In [ ]:
def best_f1_threshold(y_true, proba):
    prec, rec, thresholds = precision_recall_curve(y_true, proba)
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    idx = np.argmax(f1s[:-1])
    return thresholds[idx], f1s[idx]

lr_th, lr_f1 = best_f1_threshold(y_val, lr_proba_val)
rf_th, rf_f1 = best_f1_threshold(y_val, rf_proba_val)

print(f'LR  — melhor threshold (F1): {lr_th:.4f} (F1={lr_f1:.4f})')
print(f'RF  — melhor threshold (F1): {rf_th:.4f} (F1={rf_f1:.4f})')

print()
lr_mbest, _, lr_cmbest = evaluate_model(lr_pipe, X_val, y_val, 'Logistic Regression (best th)', lr_th)
print()
rf_mbest, _, rf_cmbest = evaluate_model(rf, X_val, y_val, 'Random Forest (best th)', rf_th)


## 13. Comparação dos resultados

In [ ]:
comparison = pd.DataFrame({
    'LR (th=0.50)': lr_m05,
    'LR (best th)': lr_mbest,
    'RF (th=0.50)': rf_m05,
    'RF (best th)': rf_mbest
}).T

print('Comparação na validação:')
print(comparison.round(4))


In [ ]:
# Importância das features (Random Forest)
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Importância das features (Random Forest):')
print(imp.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
imp.plot(kind='barh', ax=ax, color='#3498db')
ax.set_xlabel('Importância')
ax.set_title('Importância das Features — Random Forest')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Curva Precision-Recall
prec_lr, rec_lr, _ = precision_recall_curve(y_val, lr_proba_val)
prec_rf, rec_rf, _ = precision_recall_curve(y_val, rf_proba_val)

plt.figure(figsize=(7, 5))
plt.plot(rec_lr, prec_lr, label=f'Logistic Regression (PR-AUC={lr_m05["pr_auc"]:.3f})')
plt.plot(rec_rf, prec_rf, label=f'Random Forest (PR-AUC={rf_m05["pr_auc"]:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall (Validação)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 14. Conclusões e limitações

### O que os resultados mostram

- O baseline atinge ~91,5% de acurácia apenas prevendo “entregue”. Isso torna a acurácia isolada inadequada.
- Na validação, com threshold ajustado para maximizar F1:
  - **Regressão Logística** tende a maior recall (identifica mais cancelamentos) e gera mais falsos positivos.
  - **Random Forest** tende a maior precision e F1, com menos alertas incorretos, porém deixa passar mais cancelamentos.
- PR-AUC e ROC-AUC próximos de valores modestos indicam capacidade discriminativa limitada com as features atuais.
- `subtotal`, `estimated_delivery_min`, `order_weekday` e `order_hour` concentram a importância no Random Forest. `delivery_fee` teve importância zero neste experimento.

### Limitações

1. Dados **sintéticos** — não representam operação real.
2. Amostra pequena (2.000 registros; 172 cancelamentos).
3. Desbalanceamento forte.
4. Resultados ainda modestos para uso operacional.
5. O conjunto de teste já foi inspecionado em execuções anteriores da Regressão Logística; para uma avaliação “intocada” seria necessário novo split ou dados externos.
6. Features limitadas ao que está disponível no momento do pedido.

### Próximos passos sugeridos

- Avaliar no conjunto de teste com o threshold fixado na validação (e documentar que o teste já foi visto historicamente).
- Testar outras estratégias de desbalanceamento (SMOTE com cuidado, undersampling, etc.).
- Engenharia de atributos adicional e encodings controlados.
- Calibração de probabilidades.

---

**Nota de reprodução:** execute as células em ordem. Coloque `orders.csv` em `data/orders.csv`.  
Resultados numéricos podem variar levemente entre ambientes, mas a ordem de grandeza e as conclusões devem se manter.
